# Text Data - where to start?

Er zijn verschillende packages waarmee je documenten kunt analyseren.

Hieronder heb ik twee voorbeelden van Scikitlearn gepakt, waarbij een aantal documenten worden gevectoriseerd via Bag of Words. 

Het eerste voorbeeld is simpel en geeft je een beeld van hoe je de bag of words methode toepast.

Voor het tweede voorbeeld wordt k-means toegepast om de documenten te clusteren op basis van similarity in de word frequencies. Dit geeft je alvast een inkijkje in wat je straks zelf kunt gaan doen met machine learning. Alle code staat in één cell, daaronder laat ik zien wat er zoal in verschillende objecten gebeurt.

## Simpel voorbeeld

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
corpus = [
    'This is the first document.',
    'This document is the second document.',
    'And this is the third one.',
    'Is this the first document?',
    ]
vectorizer0 = TfidfVectorizer()  # opzetten van de vectorizer
X0 = vectorizer0.fit_transform(corpus)  # toepassen van de vectorizer op document `corpus`
print(vectorizer0.get_feature_names())  # opvragen van de woorden die als features zijn meegenomen
print(X0.shape)


## Clustering text documents using k-means


This is an example showing how the scikit-learn can be used to cluster
documents by topics using a bag-of-words approach. This example uses
a scipy.sparse matrix to store the features instead of standard numpy arrays.

Two feature extraction methods can be used in this example:

  - TfidfVectorizer uses a in-memory vocabulary (a python dict) to map the most
    frequent words to features indices and hence compute a word occurrence
    frequency (sparse) matrix. The word frequencies are then reweighted using
    the Inverse Document Frequency (IDF) vector collected feature-wise over
    the corpus.

  - HashingVectorizer hashes word occurrences to a fixed dimensional space,
    possibly with collisions. The word count vectors are then normalized to
    each have l2-norm equal to one (projected to the euclidean unit-ball) which
    seems to be important for k-means to work in high dimensional space.

    HashingVectorizer does not provide IDF weighting as this is a stateless
    model (the fit method does nothing). When IDF weighting is needed it can
    be added by pipelining its output to a TfidfTransformer instance.

Two algorithms are demoed: ordinary k-means and its more scalable cousin
minibatch k-means.

Additionally, latent semantic analysis can also be used to reduce
dimensionality and discover latent patterns in the data.

It can be noted that k-means (and minibatch k-means) are very sensitive to
feature scaling and that in this case the IDF weighting helps improve the
quality of the clustering by quite a lot as measured against the "ground truth"
provided by the class label assignments of the 20 newsgroups dataset.

This improvement is not visible in the Silhouette Coefficient which is small
for both as this measure seem to suffer from the phenomenon called
"Concentration of Measure" or "Curse of Dimensionality" for high dimensional
datasets such as text data. Other measures such as V-measure and Adjusted Rand
Index are information theoretic based evaluation scores: as they are only based
on cluster assignments rather than distances, hence not affected by the curse
of dimensionality.

Note: as k-means is optimizing a non-convex objective function, it will likely
end up in a local optimum. Several runs with independent random init might be
necessary to get a good convergence.


In [ ]:
%matplotlib inline

In [ ]:
# Author: Peter Prettenhofer <peter.prettenhofer@gmail.com>
#         Lars Buitinck
# License: BSD 3 clause
from sklearn.datasets import fetch_20newsgroups
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from sklearn import metrics

from sklearn.cluster import KMeans, MiniBatchKMeans

import logging
from optparse import OptionParser
import sys
from time import time

import numpy as np


# Display progress logs on stdout
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)s %(message)s')

# parse commandline arguments
op = OptionParser()
op.add_option("--lsa",
              dest="n_components", type="int",
              help="Preprocess documents with latent semantic analysis.")
op.add_option("--no-minibatch",
              action="store_false", dest="minibatch", default=True,
              help="Use ordinary k-means algorithm (in batch mode).")
op.add_option("--no-idf",
              action="store_false", dest="use_idf", default=True,
              help="Disable Inverse Document Frequency feature weighting.")
op.add_option("--use-hashing",
              action="store_true", default=False,
              help="Use a hashing feature vectorizer")
op.add_option("--n-features", type=int, default=10000,
              help="Maximum number of features (dimensions)"
                   " to extract from text.")
op.add_option("--verbose",
              action="store_true", dest="verbose", default=False,
              help="Print progress reports inside k-means algorithm.")

print(__doc__)
op.print_help()


def is_interactive():
    return not hasattr(sys.modules['__main__'], '__file__')


# work-around for Jupyter notebook and IPython console
argv = [] if is_interactive() else sys.argv[1:]
(opts, args) = op.parse_args(argv)
if len(args) > 0:
    op.error("this script takes no arguments.")
    sys.exit(1)


# #############################################################################
# Load some categories from the training set
categories = [
    'alt.atheism',
    'talk.religion.misc',
    'comp.graphics',
    'sci.space',
]
# Uncomment the following to do the analysis on all the categories
# categories = None

print("Loading 20 newsgroups dataset for categories:")
print(categories)

dataset = fetch_20newsgroups(subset='all', categories=categories,
                             shuffle=True, random_state=42)

print("%d documents" % len(dataset.data))
print("%d categories" % len(dataset.target_names))
print()

labels = dataset.target
true_k = np.unique(labels).shape[0]

print("Extracting features from the training dataset "
      "using a sparse vectorizer")
t0 = time()
if opts.use_hashing:
    if opts.use_idf:
        # Perform an IDF normalization on the output of HashingVectorizer
        hasher = HashingVectorizer(n_features=opts.n_features,
                                   stop_words='english', alternate_sign=False,
                                   norm=None)
        vectorizer = make_pipeline(hasher, TfidfTransformer())
    else:
        vectorizer = HashingVectorizer(n_features=opts.n_features,
                                       stop_words='english',
                                       alternate_sign=False, norm='l2')
else:
    vectorizer = TfidfVectorizer(max_df=0.5, max_features=opts.n_features,
                                 min_df=2, stop_words='english',
                                 use_idf=opts.use_idf)
X = vectorizer.fit_transform(dataset.data)

print("done in %fs" % (time() - t0))
print("n_samples: %d, n_features: %d" % X.shape)
print()

if opts.n_components:
    print("Performing dimensionality reduction using LSA")
    t0 = time()
    # Vectorizer results are normalized, which makes KMeans behave as
    # spherical k-means for better results. Since LSA/SVD results are
    # not normalized, we have to redo the normalization.
    svd = TruncatedSVD(opts.n_components)
    normalizer = Normalizer(copy=False)
    lsa = make_pipeline(svd, normalizer)

    X = lsa.fit_transform(X)

    print("done in %fs" % (time() - t0))

    explained_variance = svd.explained_variance_ratio_.sum()
    print("Explained variance of the SVD step: {}%".format(
        int(explained_variance * 100)))

    print()


# #############################################################################
# Do the actual clustering

if opts.minibatch:
    km = MiniBatchKMeans(n_clusters=true_k, init='k-means++', n_init=1,
                         init_size=1000, batch_size=1000, verbose=opts.verbose)
else:
    km = KMeans(n_clusters=true_k, init='k-means++', max_iter=100, n_init=1,
                verbose=opts.verbose)

print("Clustering sparse data with %s" % km)
t0 = time()
km.fit(X)
print("done in %0.3fs" % (time() - t0))
print()

print("Homogeneity: %0.3f" % metrics.homogeneity_score(labels, km.labels_))
print("Completeness: %0.3f" % metrics.completeness_score(labels, km.labels_))
print("V-measure: %0.3f" % metrics.v_measure_score(labels, km.labels_))
print("Adjusted Rand-Index: %.3f"
      % metrics.adjusted_rand_score(labels, km.labels_))
print("Silhouette Coefficient: %0.3f"
      % metrics.silhouette_score(X, km.labels_, sample_size=1000))

print()


if not opts.use_hashing:
    print("Top terms per cluster:")

    if opts.n_components:
        original_space_centroids = svd.inverse_transform(km.cluster_centers_)
        order_centroids = original_space_centroids.argsort()[:, ::-1]
    else:
        order_centroids = km.cluster_centers_.argsort()[:, ::-1]

    terms = vectorizer.get_feature_names()
    for i in range(true_k):
        print("Cluster %d:" % i, end='')
        for ind in order_centroids[i, :10]:
            print(' %s' % terms[ind], end='')
        print()

Met de `dir()` functie roep ik een lijst met alle objecten in het geheugen op.

In [ ]:
dir()

`km` is het k-means model, even kijken welke settings zijn gebruikt (dit wordt later logisch als we het in het college hebben behandeld).

In [ ]:
km

Ik zie in de code hier en daar settings worden toegepast die allemaal netjes in het object `opts` zitten. Even zien:

In [ ]:
opts

Zo zie ik dat de inverse document frequency (IDF) is toegepast (besproken in college 5 deel 3) om veel voorkomende woorden eruit te filteren. Verder zijn er 10000 features, dus 10000 woorden zijn gevonden en als dimensies gekozen in de vectorisering van de documenten - elke dimensie vertegenwoordigt de TFIDF (woordfrequency x IDF) van een bepaald woord. Welke woorden eigenlijk?

In [ ]:
words = vectorizer.get_feature_names()

In [ ]:
words[:10]

Hier zitten ook veel getallen bij die alfabetisch bovenaan staan. Als we wat verderop kijken vinden we wel normale woorden:

In [ ]:
words[2500:2510]

Er zijn ook woorden weggelaten op basis van de document frequency (zie settings: `max_df = 0.5` houdt in dat woorden die in meer dan de helft van de documenten (hier > 10 documenten) voorkomen genegeerd worden. `min_df = 2` houdt in dat enkel woorden worden meegenomen die twee keer voorkomen in tenminste één document):

In [ ]:
ignored_words = list(vectorizer.stop_words_)
ignored_words.sort()

In [ ]:
ignored_words[6000:6010]

### Searching Texts
How many and which texts contain the word "Amsterdam"? Eerst eens kijken of het woord het tot de feature vector heeft geschopt (als het niet vaak genoeg voorkwam of juist te vaak, dan kan het zijn genegeerd):

In [ ]:
'amsterdam' in words

Yep, komt voor. Wat is de index?

In [ ]:
words.index('amsterdam')

Alright, in de matrix met term frequencies (of eigenlijk TFIDFs) heeft Amsterdam dus z'n eigen kolom:

In [ ]:
X[:,819]

X is een 'sparse' matrix en heeft dus veel niet gevulde plekken in deze kolom. Enkel 6 rijen zijn gevuld, wat inhoudt dat het woord 'Amsterdam' in zes documenten voorkomt. Welke?

In [ ]:
X[:,819].nonzero()

Hoeveel documenten zijn er eigenlijk?

In [ ]:
len(dataset.data)

Laten we een text pakken waar Amsterdam in staat:

In [ ]:
dataset.data[1408]

Lijkt op een email van een wetenschapper aan de UvA!